## Figure 1 Generation

### The goal of this notebook is to produce a plot for Figure 1. 

General Schema:

For a given climate variable, say, tas, plot its global average value over time from 2015 to 2100, 1 thin line per average of ensembles per model and one thick line per average of all models, for both a non net-zero scenario and a net-zero scenario.

Outline sample early and late (non-net zero and net zero) periods.

Label accordingly.

### Imports and File Stitching

In [1]:
# Imports
import os
import re
import numpy as np
import netCDF4 as nc
from netCDF4 import Dataset
import matplotlib.pyplot as plt

# ----File Stitching----
# If in prep folder, cd back to base repository folder
if os.path.basename(os.getcwd()) == "prep":
    os.chdir('../..')

# Control which user's files are accessed
match_sk = 'sophiekim'
match_hc = 'hayeonchung'
match_ck = 'Caroline'
match_st = 'student'
path_str = os.getcwd()
print(path_str)
if re.search(match_sk, path_str):
    os.chdir("/Users/sophiekim/Desktop/2_research/MamalakisResearch") 
    user = match_sk
    print("Sophie Kim recognized as user.")
elif re.search(match_hc, path_str):
    os.chdir("/Users/hayeonchung/Downloads/Mamalakis Graduate Research/MamalakisResearch") 
    user = match_hc
    print("Hayeon Chung recognized as user.")
elif re.search(match_ck, path_str):
    os.chdir("/Users/Caroline/Desktop/school/MamalakisResearch") 
    user = 'carolinekranefuss'
    print("Caroline Kranefuss recognized as user.")
elif re.search(match_st, path_str):
    os.chdir("/Users/student/Desktop/mamalakis_research/MamalakisResearch")
    user = match_st
    print("Student recognized as user.")
else:
    print("User not recognized. Please manually change directory.")

c:\Users\student\Desktop\mamalakis_research\MamalakisResearch
Student recognized as user.


In [2]:
%%capture
%run "get_cnn_tensors.ipynb" 

In [3]:
# Assign base path
base_path = os.getcwd()

# All users should have locally loaded 'data' folder
data_path = base_path + '/data/'

In [4]:
# initializing model list variable to call in function
model_list = [
    "CNRM_ESM2-1_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MIROC6_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MPI-ESM1-2-LR_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MRI-ESM2-0_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "UKESM1-0-LL_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
]

model_paths = [data_path + model for model in model_list]

model_paths

['c:\\Users\\student\\Desktop\\mamalakis_research\\MamalakisResearch/data/CNRM_ESM2-1_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc',
 'c:\\Users\\student\\Desktop\\mamalakis_research\\MamalakisResearch/data/MIROC6_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc',
 'c:\\Users\\student\\Desktop\\mamalakis_research\\MamalakisResearch/data/MPI-ESM1-2-LR_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc',
 'c:\\Users\\student\\Desktop\\mamalakis_research\\MamalakisResearch/data/MRI-ESM2-0_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc',
 'c:\\Users\\student\\Desktop\\mamalakis_research\\MamalakisResearch/data/UKESM1-0-LL_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc']

In [57]:
def convert_units(varname: str, x: np.ndarray):
    """
    varname: index number from the list of variables so get_data func can convert units 
    x: data that needs units changed (raw x data) in get_data func 
    """
    if varname in {"tas", "tasmax", "tasmin"}:
        # kelvin to celsius
        return x - 273.15, "$^{\circ}$C"
    if varname == "pr":
        # kg/(m2*s) to mm/day
            # 1kg/m2 = 1 mm 
        return x * 86400.0, "mm/day"
    if varname == "psl":
        # pascals to hpa
        return x / 100.0, "hPa"
    
    # these don't need to be converted -- just adding the units 
    if varname == "sfcWind":
        return x, "m/s"
    if varname == "mrsos":
        return x, "kg/m$^{2}$"
    return x, "unknown"

In [ ]:
def var_mthly_avgs(model, scenario='ssp119', var="tas"):

    # Define order of variables in data loaded to obtain index
    var_list = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind", "mrsos"]
    var_index = 0 # Get the index don't hardcode for tas
    
    ### Transpose so lats and longs make sense?

    # Months range from 0 to 1032 ### CAN'T GET IT TO 1032 BC INDEX RUNS OUT??
    months = np.arange(1,1032,1)

    # Make an array for each ensemble, 5 ensembles per model
    ensembles = [[],[],[],[],[]]

    # loading the data for specific model 
    with nc.Dataset(model) as ds:

        # For each ensemble, 
        for i, ens in ensembles:
            
            # Take the average of all global values for the variable per month and append to the appropriate ensembles array
            for month in months:

                # slicing dimensions for the ensemble, the specific var index and all the lon/lat dimensions for EACH time - then find average of each time 
                vals_mth_ensemble = ds[f"data_{scenario}"][ens, var_index, month, :, :] 

                global_avg_ens_mth = np.nanmean(vals_mth_ensemble)

                ensembles[i].append(global_avg_ens_mth)
        
    # Now we have an array of arrays - average global temp/precip/etc for every month, repeated across each ensemble of a model
    # Take the average of those ensembles to get a model average
    # Should not have any NaN 
    ensembles = [np.array(x) for x in ensembles]
    var_over_time = [np.mean(k) for k in zip(*ensembles)]

    # Do unit conversion
    var_over_time, _ = convert_units("tas", ensembles)
    
    # Return an np array of the average global value for the chosen variable for each month in a model, with all ensembles for a model averaged
    return var_over_time 

var_mthly_avgs(model_paths[0])

ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
with nc.Dataset(full_path) as ds:
            # loading the data for specific model 
                # slicing dimensions for the ensembles, the specific var indices and all the times/lon/lat dimensions
            data_all_vars = ds[f"data_{scenario}"][:, var_indices, :, :, :] 


            # starting unit conversions for ALL variables
            # looping through all 7 vars or the selected vars only 
                # using enumerate to keep track of the index of var and what the var is in the var_list defined above
                    # idx to keep track of what slice of the var dimension 
                    # var_name so that convert_units can be called correctly 
            for idx, var_name in enumerate(selected_vars):
                # converting var to relevant unit from func -- slicing the relevant var one at a time (getting all the info for all the other dimensions, just associated with that certain var)
                    # returns the converted numbers and string that gives converted unit
                converted_data, _ = convert_units(var_name, data_all_vars[:, idx, :, :, :])
                 # taking all the converted data and making it the 'all data' version for the var index 
                data_all_vars[:, idx, :, :, :] = converted_data

In [ ]:
def plot_comparison_grid(early_map, late_map, main_title):

    
    # transposing everything so that lat and longs make sense 
    maps = [early_map.T, late_map.T, signal.T]
    # gives titles for the 3 subplots 
    titles = ["early period avg", "late period avg", "difference (late - early period)"]
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    plt.suptitle(main_title, fontsize=18, fontweight='bold', y=1.05)
    
    # loop runs three times for early, late and difference 
    for j in range(3):
        # for the early and late plots 
        if j < 2:
            # calculate min and max vals for the colorbar looking at the min/max vals for both the early and late periods
            vmin = min(np.nanmin(maps[0]), np.nanmin(maps[1]))
            vmax = max(np.nanmax(maps[0]), np.nanmax(maps[1]))
            cmap = 'viridis'
        else:
            # finding max values of absolute vals (whats the biggest difference found)
            sig_limit = np.nanmax(np.abs(maps[j]))
            # if there's no change, still need some sort of color bar, so if it's 0 then make it 0.1
            if sig_limit == 0: sig_limit = 0.1 
            # making min max values by using the val found above and doing neg and pos versions
            vmin, vmax = -sig_limit, sig_limit
            cmap = 'RdBu_r'
        # 
        im = axes[j].imshow(maps[j], origin='lower', cmap=cmap, 
                            vmin=vmin, vmax=vmax, aspect='auto')
        axes[j].set_title(titles[j], fontsize=14)
        cb = fig.colorbar(im, ax=axes[j])
        cb.set_label("standardized differences from baseline vals", fontsize=10)
        axes[j].set_xlabel("longitude")
        axes[j].set_ylabel("latitude")

    plt.tight_layout()
    plt.show()   

In [ ]:
def plot_var(x_data, y_data, var="tas"):
    """
    x_data: shape (samples (should be 500), 7, lat, lon)
    y_data: shape (samples (should be 500), 1)
    """


    # PLOTTING ALL VARS AVERAGED
    # averaging al the x_data tg based on variable dimension 
    multivariate_data = np.mean(x_data where index is tas, axis=1) 

    # using helper func below to plot this one 
    plot_comparison_grid(early_multi, late_multi, "multivariate plot of all 7 vars averaged together")